# 8.2 Equipment energy table

For each equipment type: mean energy use, mean unit count, and mean energy per unit
(the group's mean energy divided by its mean unit count - a ratio of averages, not an
average of each lab's own ratio, matching the equipment-share figure's convention),
using BL data for the same "matched with EL" sample used throughout. One combined table,
Overall then by faculty (Medicine or Joint, then Science) as three panels sharing the same
column headers, each closed with a "Total" row - mean total energy, mean total unit count,
and mean energy per unit across every equipment type - rather than repeating three
separate tables or reducing the footer to just a lab count.

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_equipment_energy_table import make_equipment_energy_table

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,
    na_values=[""]
)
panel = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_5.csv",
    keep_default_na=False,
    na_values=[""]
)

## (1) Prepare BL sample (matched with EL) and equipment unit counts

In [3]:
# Same "BL, sample with matched EL data" restriction as the electricity histograms
# and the equipment energy share figure
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
matched_labgroupids = labgroup_counts[labgroup_counts == 2].index

df_bl = df[df["labgroupid"].isin(matched_labgroupids) & (df["survey"] == "BL")].copy()
df_bl["MNF"] = np.where(df_bl["faculty"] == "Faculty of Science (MNF)", 1, 0)

print(f"Labs: {len(df_bl)} (Medicine or Joint: {(df_bl['MNF']==0).sum()}, Science: {(df_bl['MNF']==1).sum()})")

Labs: 109 (Medicine or Joint: 32, Science: 77)


In [4]:
# Column -> display label, for all 11 equipment types (matches the labelling used in
# 3_8_hist_equipment.rmd and the equipment energy share figure)
equipment_labels = {
    "it":        "IT Equipment",
    "fc":        "Fume Cupboard",
    "freezer":   "Freezer",
    "ult":       "ULT Freezer",
    "fridge":    "Fridge",
    "incubator": "CO2 Incubator",
    "microbio":  "Microbiological Safety Cabinet",
    "glassware": "Glassware Drying Cabinet",
    "bath":      "Water Bath",
    "cryostat":  "Cryostat",
    "heater":    "Block Heater",
}

In [5]:
# panel has one row per lab per survey per equipment type, "number" gives the count of
# units of that type. Sum to get total units per lab per equipment type at BL, then join
# onto every matched-sample lab (filling 0 for labs with none of that type).
panel_bl = panel[(panel["labgroupid"].isin(matched_labgroupids)) & (panel["survey"] == "BL")]
equip_counts = panel_bl.groupby(["labgroupid", "equipment"])["number"].sum().reset_index(name="count")

for eq in equipment_labels:
    colname = f"count_{eq}"
    eq_sub = equip_counts[equip_counts["equipment"] == eq][["labgroupid", "count"]].rename(
        columns={"count": colname}
    )
    df_bl = df_bl.merge(eq_sub, on="labgroupid", how="left")
    df_bl[colname] = df_bl[colname].fillna(0)
    df_bl[f"annual_electricity_{eq}"] = pd.to_numeric(df_bl[f"annual_electricity_{eq}"], errors="raise")

## (2) Compute means and fix row order

Row order is fixed by the Overall group's mean energy (descending) - the same ordering used
in the equipment energy share figure - and kept the same across the by-faculty tables too,
so the same equipment type sits in the same row position in every table.

In [6]:
def equipment_means(d):
    rows = []
    for eq, label in equipment_labels.items():
        rows.append({
            "label": label,
            "mean_energy": d[f"annual_electricity_{eq}"].mean(),
            "mean_units": d[f"count_{eq}"].mean(),
        })
    return pd.DataFrame(rows)

means_overall = equipment_means(df_bl)
row_order = means_overall.sort_values("mean_energy", ascending=False)["label"].tolist()

def reorder(means_df):
    return means_df.set_index("label").loc[row_order].reset_index()

means_overall = reorder(means_overall)
means_medicine = reorder(equipment_means(df_bl[df_bl["MNF"] == 0]))
means_science = reorder(equipment_means(df_bl[df_bl["MNF"] == 1]))

## (3) Build and save tables

In [7]:
count_cols = [f"count_{eq}" for eq in equipment_labels]
df_bl["count_total"] = df_bl[count_cols].sum(axis=1)

def make_panel(d, label):
    return {
        "label": label,
        "df": reorder(equipment_means(d)),
        "total_energy": d["annual_electricity_total"].mean(),
        "total_units": d["count_total"].mean(),
    }

panels = [
    make_panel(df_bl, "Overall"),
    make_panel(df_bl[df_bl["MNF"] == 0], "Medicine or Joint"),
    make_panel(df_bl[df_bl["MNF"] == 1], "Science"),
]

table = make_equipment_energy_table(panels)

out_dir = config.OUTPUT / "10_Equipment_Tables"
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "equipment_energy_table.tex"
_ = table_path.write_text(table)
print(f"Saved: {table_path}")

Saved: /Users/drutna/Dropbox/Apps/Overleaf/UZH Decarb Pre-Analysis Plan/2_Output/10_Equipment_Tables/equipment_energy_table.tex


In [8]:
print(table)

\begin{tabular}{@{}L{6.5cm}C{3.2cm}C{3.2cm}C{3.2cm}}
\hline
\addlinespace[0.2cm]
 & Mean Energy (kWh) & Mean Units & Mean Energy per Unit (kWh) \\
\hline
\addlinespace[0.2cm]
\multicolumn{4}{l}{\textit{Overall}} \\
\addlinespace[0.1cm]
Fume Cupboard & $ 8{,}877.0$ & $ 3.60$ & $ 2{,}468.3$ \\
\addlinespace[0.1cm]
IT Equipment & $ 5{,}622.6$ & $ 14.97$ & $ 375.5$ \\
\addlinespace[0.1cm]
ULT Freezer & $ 3{,}113.0$ & $ 0.65$ & $ 4{,}779.0$ \\
\addlinespace[0.1cm]
Freezer & $ 1{,}576.2$ & $ 3.30$ & $ 477.2$ \\
\addlinespace[0.1cm]
Fridge & $ 1{,}286.6$ & $ 3.56$ & $ 361.4$ \\
\addlinespace[0.1cm]
CO2 Incubator & $ 488.7$ & $ 0.89$ & $ 549.2$ \\
\addlinespace[0.1cm]
Cryostat & $ 390.2$ & $ 0.08$ & $ 4{,}725.6$ \\
\addlinespace[0.1cm]
Microbiological Safety Cabinet & $ 346.3$ & $ 0.75$ & $ 460.4$ \\
\addlinespace[0.1cm]
Glassware Drying Cabinet & $ 152.1$ & $ 0.20$ & $ 753.5$ \\
\addlinespace[0.1cm]
Water Bath & $ 100.0$ & $ 1.26$ & $ 79.6$ \\
\addlinespace[0.1cm]
Block Heater & $ 73.1$ & $ 1